In [ ]:
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

path = Path("tide_ribamar_2026-08-15_to_10-04_UTC.csv")
lines = path.read_text(encoding="utf-8-sig").splitlines()

header = next(
    index for index, line in enumerate(lines)
    if "stime" in line and "slevel" in line
)

data = pd.read_csv(
    StringIO("\n".join(lines[header:])),
    sep=None,
    engine="python",
)
data.columns = data.columns.str.strip().str.lstrip("#").str.strip()

print("Columns:", data.columns.tolist())
data["stime"] = pd.to_datetime(data["stime"], utc=True, errors="coerce")
data["slevel"] = pd.to_numeric(data["slevel"], errors="coerce")
data = data.dropna(subset=["stime"]).sort_values("stime")

figure, axis = plt.subplots(figsize=(14, 5))
axis.plot(data["stime"], data["slevel"], linewidth=0.7)
axis.set(
    title="São José de Ribamar — Water level",
    xlabel="Date (UTC)",
    ylabel="Water level (m, station reference)",
)
axis.grid(alpha=0.3)
figure.autofmt_xdate()
figure.tight_layout()
figure.savefig("tide_ribamar_plot.png", dpi=200)
plt.show()


In [ ]:
figure_dir = Path("../figures/Belém")
scenes = pd.DataFrame(
    {
        "scene": sorted(
            path.stem.removeprefix("pauli_")
            for path in figure_dir.glob("pauli_*.png")
        )
    }
)
scenes["time_utc"] = pd.to_datetime(
    scenes["scene"].str.slice(0, 15), format="%Y%m%dT%H%M%S", utc=True
)


def water_level(time):
    position = data["stime"].searchsorted(time)
    candidates = [
        index for index in (position - 1, position) if 0 <= index < len(data)
    ]
    nearest = min(candidates, key=lambda index: abs(data["stime"].iloc[index] - time))
    if abs(data["stime"].iloc[nearest] - time) > pd.Timedelta("5min"):
        return float("nan")
    return float(data["slevel"].iloc[nearest])


scenes["level_m"] = scenes["time_utc"].map(water_level).round(2)

figure, axis = plt.subplots(figsize=(14, 5))
axis.plot(data["stime"], data["slevel"], linewidth=0.7, zorder=1)
for time in scenes["time_utc"]:
    axis.axvline(time, color="red", alpha=0.35, linewidth=0.8, zorder=2)
axis.scatter(
    scenes["time_utc"],
    scenes["level_m"],
    color="red",
    s=18,
    zorder=3,
    label="Scene acquisition",
)
axis.set(
    title="São José de Ribamar — Water level at scene acquisition",
    xlabel="Date (UTC)",
    ylabel="Water level (m, station reference)",
)
axis.grid(alpha=0.3)
axis.legend(loc="upper right")
figure.autofmt_xdate()
figure.tight_layout()
figure.savefig("tide_ribamar_scenes.png", dpi=200)
plt.show()

scenes


In [ ]:
delays_h = [0, 1, 2, 3]
for hours in delays_h:
    scenes[f"level_{hours}h"] = (
        (scenes["time_utc"] - pd.Timedelta(hours=hours)).map(water_level).round(2)
    )

figure, axis = plt.subplots(figsize=(14, 5))
colors = ["0.45", "C0", "C1", "C3"]
for time in scenes["time_utc"]:
    axis.axvline(time, color="0.75", linewidth=0.6, zorder=0)
for hours, color in zip(delays_h, colors):
    axis.scatter(
        scenes["time_utc"],
        scenes[f"level_{hours}h"],
        s=28,
        color=color,
        zorder=3,
        label=f"{hours} h before acquisition",
    )
for _, scene in scenes.iterrows():
    axis.plot(
        [scene["time_utc"]] * len(delays_h),
        [scene[f"level_{hours}h"] for hours in delays_h],
        color="0.7",
        linewidth=0.8,
        zorder=2,
    )
axis.set(
    title="Gauge level 0–3 h before each scene (travel time toward the forest)",
    xlabel="Acquisition (UTC)",
    ylabel="Water level (m, station reference)",
)
axis.grid(alpha=0.3, axis="y")
axis.legend(loc="upper right")
figure.autofmt_xdate()
figure.tight_layout()
figure.savefig("tide_ribamar_delay.png", dpi=200)
plt.show()

scenes[["scene", "time_utc", "level_0h", "level_1h", "level_2h", "level_3h"]]
